In [1]:
# Import zipfile module to work with ZIP files
import zipfile

# Define the path of the downloaded zip file
zip_path = "spam.zip"   # change name if your zip file name is different

# Define the folder where files will be extracted
extract_path = "."   # "." means extract in the current directory (Jupyter folder)

# Open the zip file in read mode
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    
    # Extract all contents of the zip file to the specified folder
    zip_ref.extractall(extract_path)

# Print confirmation message after extraction
print("Dataset unzipped successfully!")


Dataset unzipped successfully!


In [2]:
# Import os module to interact with the operating system
import os

# List all files in the current directory
os.listdir()


['.ipynb_checkpoints',
 'archive.zip',
 'data',
 'Expense_Tracker.ipynb',
 'House_Predict.ipynb',
 'readme',
 'SMSSpamCollection',
 'spam.zip',
 'Untitled.ipynb']

In [3]:
# Import pandas library for handling tabular data
import pandas as pd

# Read the SMSSpamCollection file into a DataFrame
# sep="\t" tells pandas that columns are separated by a tab
# names assigns column names because file has no header row
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    names=["label", "message"]
)

# Display the first 5 rows to confirm data loaded correctly
df.head()


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Show total number of rows and columns
df.shape

# Show column data types and memory info
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [5]:
# Convert 'ham' and 'spam' text labels into numerical values
# 'ham' is mapped to 0 (Not Spam)
# 'spam' is mapped to 1 (Spam)
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

# Display first 5 rows to confirm label conversion
df.head()


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# Count how many spam and non-spam messages exist
df['label'].value_counts()


0    4825
1     747
Name: label, dtype: int64

In [7]:
# Select the 'message' column as input feature (X)
# This is the text data that the model will analyze
X = df['message']

# Select the 'label' column as target variable (y)
# This is what the model should predict (0 = Not Spam, 1 = Spam)
y = df['label']


In [8]:
# Check first 3 messages
X.head(3)

# Check first 3 labels
y.head(3)


0    0
1    0
2    1
Name: label, dtype: int64

In [9]:
# Import train_test_split function to divide data
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing parts
# X_train -> messages used to train the model
# X_test  -> messages used to test the model
# y_train -> labels used to train the model
# y_test  -> labels used to test the model
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                 # input features and target labels
    test_size=0.2,        # 20% data is kept for testing
    random_state=42       # ensures same split every time we run
)


In [10]:
# Print number of samples in each set
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 4457
Testing samples: 1115


In [11]:
# Import CountVectorizer to convert text into numerical feature vectors
from sklearn.feature_extraction.text import CountVectorizer

# Create a CountVectorizer object
# It converts each word into a feature based on word frequency
vectorizer = CountVectorizer()

# Learn the vocabulary from training data and transform text into numbers
# fit_transform learns words + converts text into numeric matrix
X_train_vectorized = vectorizer.fit_transform(X_train)

# Transform test data using the SAME vocabulary learned from training data
# We do NOT use fit_transform here to avoid data leakage
X_test_vectorized = vectorizer.transform(X_test)


In [12]:
# View number of features (unique words learned)
X_train_vectorized.shape


(4457, 7702)

In [13]:
# See the actual words learned by the vectorizer
vectorizer.get_feature_names_out()[:20]   # first 20 words


array(['00', '000', '000pes', '008704050406', '0089', '0121',
       '01223585236', '01223585334', '02', '0207', '02072069400',
       '02073162414', '02085076972', '021', '03', '04', '0430', '05',
       '050703', '0578'], dtype=object)

In [14]:
# Import Multinomial Naive Bayes algorithm
# This variant is best suited for text data (word counts)
from sklearn.naive_bayes import MultinomialNB

# Create the Naive Bayes model object
model = MultinomialNB()

# Train the model using vectorized training data
# The model learns word patterns for spam and non-spam
model.fit(X_train_vectorized, y_train)


MultinomialNB()

In [15]:
# Use the trained Naive Bayes model to predict labels
# This predicts whether each test message is spam (1) or not spam (0)
y_pred = model.predict(X_test_vectorized)


In [16]:
# View first 10 predictions
y_pred[:10]

# View actual labels for comparison
y_test.values[:10]


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)

In [17]:
# Import accuracy_score to calculate model accuracy
from sklearn.metrics import accuracy_score

# Compare actual labels (y_test) with predicted labels (y_pred)
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy of the Naive Bayes model
print("Model Accuracy:", accuracy)


Model Accuracy: 0.9919282511210762


In [18]:
# Create a list of new messages to test the model
# These messages were NOT part of training or testing data
test_messages = [
    "Win free cash now",
    "Hey are we going to college tomorrow?",
    "Congratulations you have won a prize",
    "Let's have dinner tonight",
    "Urgent! claim your reward now"
]

# Convert these new text messages into numerical form
# We use transform(), NOT fit_transform(), to keep vocabulary same
test_messages_vectorized = vectorizer.transform(test_messages)

# Predict spam (1) or not spam (0) for the new messages
predictions = model.predict(test_messages_vectorized)

# Loop through messages and predictions together
for message, prediction in zip(test_messages, predictions):
    
    # Print message and its predicted label
    # If prediction is 1 → Spam, else → Not Spam
    print(message, "->", "Spam" if prediction == 1 else "Not Spam")


Win free cash now -> Spam
Hey are we going to college tomorrow? -> Not Spam
Congratulations you have won a prize -> Spam
Let's have dinner tonight -> Not Spam
Urgent! claim your reward now -> Spam
